# L59 · PyTorch Tensor 基础 — 与 NumPy 互转、device、requires_grad

**目标**：掌握 `torch.Tensor` 的基础操作，理解和 numpy 的异同，为 GPU 训练做准备

🔗 **Aurora 连接**：Month 2 数据集加载器把 `aurora.audio.mfcc.mfcc()` 输出的 numpy MFCC 特征矩阵转成 `torch.Tensor` 送入神经网络；本节的 `mel_to_batch()` 就是那个转换层。

← **上一课**　[L58 · 训练循环](L58_training_loop.ipynb)

> 上节课学习了 **训练循环**：loss 计算、optimizer.step、收敛曲线，拟合 makemoons 数据集。  
> 本课将探讨 **PyTorch Tensor 基础**。

## 本课剧情：NumPy 为什么"到了GPU门口被拦下来"？

你已经用 NumPy 实现了 MFCC、FFT、Mel——它们在 CPU 上完美运行。但把它们喂给 PyTorch 神经网络时，会遇到一道坎：

**NumPy 不知道"梯度"是什么，也不能住在 GPU 里。**

PyTorch 的 `Tensor` 是升级版 ndarray，增加了三件武器：

| 能力 | NumPy ndarray | PyTorch Tensor |
|---|---|---|
| 存数值 | ✅ | ✅ |
| GPU 加速 | ❌ | ✅ (`device='cuda'`) |
| 自动梯度 | ❌ | ✅ (`requires_grad=True`) |
| 与 NumPy 互转 | — | `from_numpy` / `.numpy()` |

**关键互转**：
```python
arr = np.array([1.0, 2.0, 3.0])          # NumPy float64
t = torch.from_numpy(arr)                  # 零拷贝，共享内存！
arr[0] = 99.0                              # 改 arr → t 也变！
t2 = torch.tensor(arr, dtype=torch.float32)  # 复制，float32
```

**Aurora 连接**：`mel_to_batch()` 把 NumPy MFCC 列表转为 `(B, 1, T, n_mels)` 的 `float32` Tensor，这就是 CNN 的标准输入格式（batch, channel, height, width）。

本节任务：理解 Tensor 三大属性（dtype/shape/device），实现 `mel_to_batch()`。

In [ ]:
try:
    import torch
    import numpy as np
    from aurora.audio.mfcc import mfcc
    HAS_TORCH = True
except ImportError as e:
    HAS_TORCH = False
    print(f'⚠️ {e}；PyTorch 相关 cell 将跳过')


## 1. Tensor vs ndarray：dtype · shape · device

| 属性 | numpy | torch |
|------|-------|-------|
| 类型 | `arr.dtype` | `t.dtype`（`torch.float32` 等）|
| 形状 | `arr.shape` → tuple | `t.shape` → `torch.Size` |
| 设备 | 始终 CPU | `t.device`：`cpu` 或 `cuda:0` |

dtype 默认差异：`np.array([1.0])` 是 `float64`，`torch.tensor([1.0])` 是 `float32`。
神经网络权重通常用 `float32`，混合精度训练（mixed precision training）用 `float16` 或 `bfloat16`。

互转：`torch.from_numpy(arr)` 共享内存（零拷贝），`t.numpy()` 反向转回（仅 CPU Tensor）。

In [ ]:
arr = np.array([[1.0, 2.0], [3.0, 4.0]])   # float64
t   = torch.from_numpy(arr)                 # 共享内存，dtype=float64
t32 = t.float()                             # 转 float32

print('numpy dtype :', arr.dtype)
print('torch dtype :', t.dtype, '->', t32.dtype)
print('shape        :', t.shape)
print('device       :', t.device)

# 修改 arr 会影响 t（共享内存）
arr[0, 0] = 99.0
print('t[0,0] after arr mutation:', t[0, 0].item())  # 99.0

## 2. 自动梯度追踪：`requires_grad` 与计算图

`requires_grad=True` 告诉 PyTorch 对该 Tensor 的所有操作都构建有向无环图（DAG）。
调用 `.backward()` 时，PyTorch 沿 DAG 反向传播，把梯度累积（gradient accumulation）到每个叶子节点的 `.grad`。

手算验证：设 `x = 3.0`，`y = x**2 + 2*x`，则 `dy/dx = 2x + 2 = 8`。
```
y = x**2 + 2*x
dy/dx = 2*x + 2  (在 x=3 时 = 8)
```

注意：`.backward()` 默认只对**标量**调用。对非标量输出需先 `.sum()` 或传入 `gradient` 参数。
叶子节点（用户创建、非运算结果）才会积累 `.grad`；中间节点梯度默认不保留（`retain_graph` 除外）。

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x          # y = 9 + 6 = 15
y.backward()            # dy/dx = 2x+2 = 8

print('y      =', y.item())       # 15.0
print('x.grad =', x.grad.item()) # 8.0  ← 和手算一致

## 3. 常用张量（tensor）操作

**形状变换**
- `view(shape)` / `reshape(shape)`：重排维度，总元素数不变。`view` 要求内存连续，`reshape` 自动处理。
- `squeeze(dim)` / `unsqueeze(dim)`：删除/插入大小为 1 的维度。

**拼接**
- `torch.cat([a, b], dim=0)`：沿已有维度拼接，不新增维度。
- `torch.stack([a, b], dim=0)`：新建一个维度再拼接，要求各 Tensor 形状完全相同。

**乘法**
- `@`（等同 `torch.matmul`）：矩阵乘法，支持批量广播。
- `torch.einsum('bi,ij->bj', x, W)`：爱因斯坦求和（Einstein summation，einsum），写出下标比反复 `transpose` 清晰。
```
einsum 规则：重复的下标代表求和（收缩），箭头右侧是输出下标。
'bi,ij->bj'：对 i 求和 → 等价于 x @ W
```

In [ ]:
# --- view / unsqueeze ---
a = torch.arange(12).float()         # shape (12,)
b = a.view(3, 4)                      # (3, 4)
c = b.unsqueeze(0)                    # (1, 3, 4)
print('view:', b.shape, '  unsqueeze:', c.shape)

# --- cat vs stack ---
x = torch.ones(2, 3)
y2 = torch.zeros(2, 3)
print('cat  dim=0:', torch.cat([x, y2], dim=0).shape)   # (4,3)
print('stack dim=0:', torch.stack([x, y2], dim=0).shape) # (2,2,3)

# --- @ 与 einsum ---
W = torch.randn(3, 5)
inp = torch.randn(4, 3)              # batch=4, in_dim=3
out_mm = inp @ W                     # (4, 5)
out_ein = torch.einsum('bi,ij->bj', inp, W)
print('matmul == einsum:', torch.allclose(out_mm, out_ein))

## 4. ✏️ 实现 `mel_to_batch(mel_list)`

**签名**：
```python
def mel_to_batch(mel_list: list[np.ndarray]) -> torch.Tensor:
    # mel_list: list of (T, n_mels) ndarray
    # 返回: (B, 1, T, n_mels) float32 Tensor
```

**4步实现路线**：

| 步骤 | 操作 | 说明 |
|---|---|---|
| 1 | `np.stack(mel_list)` | 拼成 `(B, T, n_mels)` ndarray |
| 2 | `torch.from_numpy(...).float()` | 转 Tensor，dtype → float32 |
| 3 | `.unsqueeze(1)` | 插入 channel 维 → `(B, 1, T, n_mels)` |
| 4 | 返回 | 验证 `.shape == (B, 1, T, n_mels)` |

**验收标准**：
- 输入 `[np.zeros((80, 40))] * 3` → 输出 shape `(3, 1, 80, 40)`
- `dtype == torch.float32`（不是 float64）
- 返回值是 `torch.Tensor`（不是 ndarray）

In [ ]:
def mel_to_batch(mel_list):
    """
    mel_list : list of np.ndarray, each shape (T, n_mels)
    returns  : torch.Tensor shape (B, 1, T, n_mels), dtype float32
    """
    raise NotImplementedError("TODO: stack mel_list → unsqueeze(1) → float32 Tensor")

In [ ]:
# 检查 mel_to_batch
batch = mel_to_batch([np.zeros((80, 40)) for _ in range(3)])
assert batch.shape == (3, 1, 80, 40), f'shape 错误：{batch.shape}'
assert batch.dtype == torch.float32,   f'dtype 错误：{batch.dtype}'
print('✅ mel_to_batch shape:', batch.shape, '  dtype:', batch.dtype)

## 5. 参数实验：`requires_grad` 与梯度流

实验参数与预期现象：
- `requires_grad=False`（默认）：`.grad` 保持 `None`，调用 `.backward()` 会抛 `RuntimeError`
- `requires_grad=True`：每次 `.backward()` 后 `.grad` 累加（多次调用前需 `x.grad.zero_()`）
- `torch.no_grad()` 上下文：内部操作不建计算图，推理阶段节省内存

下面验证梯度累加陷阱和 `no_grad` 的效果。

In [ ]:
# 实验 A：requires_grad=False 不会有梯度
x_ng = torch.tensor(3.0)           # requires_grad 默认 False
y_ng = x_ng ** 2
print('requires_grad=False, x_ng.grad:', x_ng.grad)  # None

# 实验 B：梯度累加陷阱
x2 = torch.tensor(3.0, requires_grad=True)
for _ in range(3):
    y2 = x2 ** 2 + 2 * x2
    y2.backward()
    # 没有 zero_() 则梯度累加
print('3次 backward 后 x2.grad:', x2.grad.item())  # 8*3 = 24（累加）

# 正确做法：每次迭代清零
x3 = torch.tensor(3.0, requires_grad=True)
for _ in range(3):
    if x3.grad is not None:
        x3.grad.zero_()
    y3 = x3 ** 2 + 2 * x3
    y3.backward()
print('清零后每次 x3.grad:', x3.grad.item())        # 8.0（正确）

# 实验 C：torch.no_grad() 不建图
x4 = torch.tensor(3.0, requires_grad=True)
with torch.no_grad():
    y4 = x4 ** 2
print('no_grad 下 y4.requires_grad:', y4.requires_grad)  # False

## 本课收束

`mel_to_batch()` 把 numpy MFCC 列表转为 `(B, 1, T, n_mels)` 的 `float32` Tensor，是 Aurora Month 2 数据集加载器与 CNN 声学模型之间的桥梁。
`torch.from_numpy` 实现零拷贝转换，`unsqueeze(1)` 插入 channel 维以满足 `Conv2d` 对 `B,C,H,W` 的要求。
`requires_grad` 机制让 Tensor 在前向传播（forward pass）时同步构建计算图，`backward()` 一键完成反向求导。
下一节（L60）将剖析 PyTorch autograd 机制：追踪 grad_fn、手动调用 backward()，理解为何 retain_graph 在有向无环计算图中不可缺少。

## ✏️ 白板挑战：Tensor 核心概念手推（目标 10 分钟）

盖上屏幕，纸上完成：

**问 1**：`torch.from_numpy(arr)` 和 `torch.tensor(arr)` 有什么区别？哪个更省内存？  
（from_numpy 共享内存（零拷贝），改 arr 会改 Tensor；tensor() 复制数据，更安全但占内存）

**问 2**：`requires_grad=True` 的 Tensor `x`，调用 `y = x**2; y.backward()` 后，`x.grad` 是什么？  
（`dy/dx = 2x`，若 x=3.0 则 x.grad=6.0）

**问 3**：`view(-1, 4)` 和 `reshape(-1, 4)` 的区别是什么？哪种更安全？  
（view 要求内存连续，不连续则报错；reshape 在可能时用 view，否则复制，更安全）

**问 4**：MFCC 特征 `(T, 40)` 要输入 2D-CNN，需要经过哪两步形状变换？  
（step1: stack B个 → (B,T,40)；step2: unsqueeze(1) → (B,1,T,40)，插入 channel 维）

**问 5**：为什么 PyTorch 默认 `float32` 而 NumPy 默认 `float64`？  
（float32 在 GPU 上速度是 float64 的 2-8×，且精度对深度学习已足够；NumPy 面向科学计算，float64 是标准）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import torch, numpy as np

# 问1：from_numpy 共享内存
arr_q = np.array([1.0, 2.0, 3.0])
t_q = torch.from_numpy(arr_q)
arr_q[0] = 99.0
assert t_q[0].item() == 99.0, "from_numpy 应共享内存"
print(f"Q1 ✅  from_numpy 共享内存：改arr[0]=99 → t[0]={t_q[0].item():.0f}")

# 问2：requires_grad backward
x_q = torch.tensor(3.0, requires_grad=True)
y_q = x_q ** 2
y_q.backward()
assert abs(x_q.grad.item() - 6.0) < 1e-6, f"x.grad={x_q.grad.item()}"
print(f"Q2 ✅  x=3.0, y=x²: dy/dx={x_q.grad.item():.1f} = 2×3 = 6")

# 问3：view vs reshape
a_q = torch.arange(12).float()
b_view = a_q.view(3, 4)
b_reshape = a_q.reshape(3, 4)
assert b_view.shape == b_reshape.shape == (3, 4)
print(f"Q3 ✅  view和reshape均得到(3,4)；view要求连续内存，reshape更宽松")

# 问4：mel_to_batch shape变换
try:
    mel_q = [np.zeros((80, 40)) for _ in range(3)]
    batch_q = mel_to_batch(mel_q)
    assert batch_q.shape == (3, 1, 80, 40), f"shape={batch_q.shape}"
    assert batch_q.dtype == torch.float32
    print(f"Q4 ✅  mel_to_batch: (80,40)×3 → {tuple(batch_q.shape)}, dtype={batch_q.dtype}")
except NotImplementedError:
    print(f"Q4 ✅  mel_to_batch: stack→(3,80,40)→from_numpy→float()→unsqueeze(1)→(3,1,80,40)（待实现）")

# 问5：float32 vs float64（概念验证）
t32 = torch.zeros(1000, dtype=torch.float32)
t64 = torch.zeros(1000, dtype=torch.float64)
assert t32.element_size() == 4
assert t64.element_size() == 8
print(f"Q5 ✅  float32={t32.element_size()}字节/元素，float64={t64.element_size()}字节，GPU效率比≈2-8×")
print("\n🎉 Tensor 基础白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l59_review = {
    "tensor_vs_ndarray":       None,  # 记住Tensor比ndarray多GPU+梯度两个能力？True/False
    "from_numpy_shared_mem":   None,  # 理解from_numpy共享内存，tensor()复制？True/False
    "requires_grad_backward":  None,  # 能手推简单函数的自动梯度？True/False
    "mel_to_batch_impl":       None,  # mel_to_batch()实现正确，shape/dtype验证通过？True/False
    "whiteboard_passed":       None,  # 白板挑战5道全部完成？True/False
}

unfilled = [k for k, v in l59_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l59_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L59 全部通关！进入 L60：autograd 机制')

---

→ **下一课**　[L60 · autograd 机制](L60_pytorch_autograd.ipynb)

> 下节课将学习 **autograd 机制**：gradfn 计算图，backward()，retaingraph 原理。